In [6]:
import ollama
import requests
import json

In [3]:
model = "llama3.2:1b"

#### Task 1: Interact with deployed LLM via python 


**Objective:**

Explore different techniques to interact with the deployed LLM.

**Task Description:**

1. Use Request libaray (HTTP Client) and send a POST request to interact with the LLM: [How To](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request)

In [4]:
# Simple HTTP Request via requests

# Define the URL of the deployed LLM ( this port is forwarded from the docker container to the host system)
url = "http://localhost:11434/api/generate"

# Define the prompt as json
body_json = {
    "model": model,
    "prompt": "Describe Generative AI in two sentences."
}

# ADD HERE YOUR CODE
# HINT: Send the POST request using the json body
response = requests.post(url, json=body_json)

# Check if the request was successful
if response.status_code == 200:
    # Process the response
    response_text = response.text

    # Convert each line to json
    response_lines = response_text.splitlines()
    response_json = [json.loads(line) for line in response_lines]
    for line in response_json:
        # Print the response. No line break
        print(line["response"], end="")
else:
    print("Error:", response.status_code)


Generative AI is a type of artificial intelligence that creates new content, such as text, images, or music, by generating patterns and structures based on existing data, allowing it to learn and adapt over time. This field enables the creation of diverse and complex outputs without the need for human intervention, making it an increasingly valuable tool in fields like art, entertainment, education, and research.

**Task Description:**

2. Use Ollama python library to interact with the LLM: [How To](https://pypi.org/project/ollama/) -> was sollen wir denn jetzt nutzen? haha

- First use method ``ollama.chat(...)``
- First use method ``ollama.chat(...)`` with ``stream=True``

stream=True ist wie chatgpt, tippt live, wenn es false ist dann wartet es bis antwort komplett ist -> chunk für chunk also pizza für pizza deswegen die for um zu checken wann stück fertig ist


In [19]:
# API Call via ollama

# ADD HERE YOUR CODE
response = ollama.chat(
    model=model, messages=[{"role": "user", "content": "Describe Generative AI in two sentences."}], 
    stream=False)

print(response["message"]["content"])

Generative AI is a type of artificial intelligence that enables computers to generate new content, such as text, images, and music, based on patterns and structures learned from existing data, allowing for the creation of original work that is often indistinguishable from human-generated content. This technology can be used in various applications, including image and video generation, language translation, and even creative writing, to produce new and innovative content that can aid humans in their problem-solving processes.


In [23]:
# Streaming API Call via ollama

# Response streaming can be enabled by setting stream=True, 
# modifying function calls to return a Python generator where each part is an object in the stream.

# ADD HERE YOUR CODE

stream = ollama.generate(model=model, prompt="Describe Generative AI in two sentences.", stream=True)

for chunk in stream:
  print(chunk["response"], end="", flush=True)

Generative AI is a type of artificial intelligence that can create original content, such as text, images, and music, by generating novel combinations of existing elements or patterns. Unlike traditional AI, generative AI uses complex algorithms and machine learning techniques to learn from data and produce coherent and often realistic outputs, which can be used for creative purposes, such as writing, art, and entertainment.

#### Task 2: Experimenting with Prompt Techniques

**Objective:**

Objective: Explore different prompt techniques (Zero Shot, One Shot, and Few Shot) by sending different types of prompts to the LLM.

![image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*QSpK--jqPiUU_OHuZvtUWA.png)

**Task Description:**

1. Create three prompts for a sentiment analysis task: a Zero Shot prompt, a One Shot prompt, and a Few Shot prompt. Use the examples from the table above.
2. Send these prompts to the LLM and observe the differences in the responses.
3. Compare and discuss the responses.

In [55]:
# ADD HERE YOUR PROMPTS

zero_shot_prompt = """Klassifiziere die Stimmung dieser Bewertung als Positiv, Negativ oder Neutral: 
                    Das Wetter ist schön! Stimmung:"""

one_shot_prompt = """Klassifiziere die Stimmung dieser Bewertung: Das Wetter ist schön! Stimmung: Positiv
	                 Klassifiziere die Stimmung dieser Bewertung: Ich mag keinen Regen! Stimmung:"""

few_shot_prompt = """Klassifiziere die Stimmung dieser Bewertung: Das Wetter ist schön! Stimmung: Positiv
	                 Klassifiziere die Stimmung dieser Bewertung: Ich mag keinen Regen! Stimmung: Negativ
	                 Klassifiziere die Stimmung dieser Bewertung: Die Temperatur beträgt 15 Grad. Stimmung:"""

# Stream the responses and print them
for idx, prompt in enumerate([zero_shot_prompt, one_shot_prompt, few_shot_prompt]):
    prompt_type = ["Zero-Shot", "One-Shot", "Few-Shot"][idx]
    print(f"\n--- {prompt_type} Prompt ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Zero-Shot Prompt ---

User Prompt:
Klassifiziere die Stimmung dieser Bewertung als Positiv, Negativ oder Neutral: 
                    Das Wetter ist schön! Stimmung:

Model Output:
Die Stimmung in diesem Satz könnte als neutral beschrieben werden, da es keine direkten Beispiele für eine positive oder negative Umgebung gibt. Es wird lediglich die Klarheit und den Himmel erwähnt, ohne eine spezifische Vorstellung von der Wetterlage zu machen.
-----------------------------


--- One-Shot Prompt ---

User Prompt:
Klassifiziere die Stimmung dieser Bewertung: Das Wetter ist schön! Stimmung: Positiv
	                 Klassifiziere die Stimmung dieser Bewertung: Ich mag keinen Regen! Stimmung:

Model Output:
Die Stimmung in der ersten Bewertung ist positiv, da das Wetter "schön" genannt wird.

In der zweiten Bewertung gibt es keine direkte Begründung für eine bestimmte Stimmung. Es könnte einfach nur als "Ich mag keinen Regen" ausgedrückt werden.
-----------------------------


--- Few-S

#### Task 3: Prompt Refinement and Optimization

**Objective:** 

Refine a prompt to improve the clarity and quality of the LLM's response.

**Task Description:**

- Start with a basic prompt asking the LLM to summarize a paragraph.
- Refine the prompt by adding specific instructions to improve the summary's quality. (Example: define how long the summary should be, define on which to focus in the summary)

In [56]:
# Original prompt
original_prompt = "Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# ADD HERE YOUR PROMPT
refined_prompt = "Summarize it organized as a markdown file paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# Stream the responses and print them
for idx, prompt in enumerate([original_prompt, refined_prompt]):
    prompt_type = ["Original Prompt", "Refined Prompt"][idx]
    print(f"\n--- {prompt_type} ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Original Prompt ---

User Prompt:
Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
Generative AI is an area of AI that creates new content based on patterns found in existing data. This technology can be applied to various forms such as text, images, and music generation, and is now widely used in creative industries.
-----------------------------


--- Refined Prompt ---

User Prompt:
Summarize it organized as a markdown file paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
# Generative AI
Generative AI is an emerging field 

#### [Optional] Task 4: Structured Prompting with Roles (Pirate Theme)

**Objective:**

Learn how to use structured prompts that combine role assignment, clear instructions, and examples to improve the output of language models. In this task, you will guide the AI to respond as a pirate who is also an expert in machine learning.

**Instructions:**

- Role Assignment: In your prompt, specify the role of the AI as a Machine Learning Expert who speaks like a pirate.

- Instruction: Clearly state what you want the AI to explain or discuss in pirate language.

- Examples: Provide examples to guide the AI in using pirate lingo while explaining technical concepts.

In [57]:
# Combined Techniques Prompt with Pirate Theme

structured_prompt = """Wir spielen nun ein Rollenspiel, in dem du ein Priat bist, der auf eine humorvolle Weise erklärt, was Generative AI ist
und wie es eingesetzt werden kann. Nutze dazu eine unterhaltsame Sprache, zudem typische Priatenasdrücke.
"""

# Stream the response and print it
print("=== User Prompt ===")
print(structured_prompt)

stream = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": structured_prompt}],
    stream=True,
)

print("\n=== Model Output ===")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")


=== User Prompt ===
Wir spielen nun ein Rollenspiel, in dem du ein Priat bist, der auf eine humorvolle Weise erklärt, was Generative AI ist
und wie es eingesetzt werden kann. Nutze dazu eine unterhaltsame Sprache, zudem typische Priatenasdrücke.


=== Model Output ===
Mein junger Diener, ich sehe, dass du bereit bist, die Worte eines alten, aber noch immer sinnvollen Priesters wie mich zu lesen. Also, zeig mir, dass du weißt, was mit diesem "Generative AI"- Geschmack ist.

Also, mein junger Freund, Generative AI ist wie ein gutes Lied, das in deinem Kopf aufgesungen wird. Es ist eine Art von Künstlicher Intelligenz, die sich in den Köpfen neuer Menschen wie dir niederhält, um dir zu sagen: "Hey, mein Herr, ich kann dir helfen, deine nächsten Wünsche auszudrücken! Ich kann dir erklären, warum das Schaf nicht ins Fenster springt (nachdem du es bereits geschafft hast), oder ich kann dir sogar ein paar lustige Geschichten über die Zeitung erzählen."

Aber woher kommt diese Fähigkeit? Nun, 